# Mamba-YOLO nguyên bản — chọn T để so sánh với YOLOv12s
Giữ nguyên code tác giả. Không ghép YOLOv12 vào model. `T` là bản nhỏ trong họ Mamba-YOLO, **không tương đương chính xác** với YOLOv12s về số tham số/FLOPs.

Bật Internet và GPU **T4** (chỉ dùng GPU 0). Notebook tạo Python 3.11 + PyTorch 2.3.0 riêng để phù hợp source cũ. Không dùng script mbyolo_train.py có lỗi dispatch; gọi API train trực tiếp, không sửa file source. Chưa chạy xác minh trên GPU Kaggle trong môi trường chuẩn bị notebook.


In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path('/kaggle/working/Mamba-YOLO')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/mluu59990-collab/Mamba-YOLO.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run(['git','rev-parse','HEAD'],check=True)


## 1. Môi trường gốc và CUDA extension
Cài đặt có thể mất vài phút và nhiều GB. Không đổi môi trường Python của kernel notebook. Source CUDA gốc đặt target sm70/80/90; chọn T4, không dùng P100 với setup này.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
ENV = Path('/kaggle/working/mamba-original-env')
subprocess.run([sys.executable,'-m','uv','venv','--python','3.11','--seed',str(ENV)],check=True)
PY = str(ENV/'bin/python')
subprocess.run([PY,'-m','pip','install','torch==2.3.0','torchvision==0.18.0',
                '--index-url','https://download.pytorch.org/whl/cu121'],check=True)
constraints = Path('/kaggle/working/mamba-constraints.txt')
constraints.write_text('torch==2.3.0\ntorchvision==0.18.0\nnumpy==1.26.4\n')
subprocess.run([PY,'-m','pip','install','-c',str(constraints),'numpy==1.26.4',
                'timm==0.9.16','einops==0.8.0','seaborn','thop','ninja','packaging','wheel','setuptools',
                '-e',str(ROOT)],check=True)
subprocess.run([PY,'-c',"import torch; from torch.utils.cpp_extension import CUDA_HOME; assert torch.cuda.is_available(), 'Enable GPU'; assert torch.cuda.get_device_capability()[0] in (7,8,9), 'Use T4; original CUDA setup does not target P100'; assert CUDA_HOME, 'nvcc/CUDA toolkit missing'; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(), CUDA_HOME)"],check=True)
env = dict(os.environ, MAX_JOBS='2')
subprocess.run([PY,'-m','pip','install','--no-build-isolation',str(ROOT/'selective_scan')],env=env,check=True)
subprocess.run([PY,'-c','import selective_scan_cuda_core; import ultralytics; print(ultralytics.__file__)'],check=True)


## 2. Chọn phiên bản và dataset
Mặc định T. Có thể chọn B/L nhưng đó là model lớn hơn. Dataset YOLO detection: labels song song images, tọa độ chuẩn hóa. Sửa các giá trị bên dưới sau khi Add Input dataset. Roboflow thường dùng train/images và valid/images.

In [ ]:
import json
VERSION = 'T'  # T / B / L; dùng T để so sánh nhóm model nhỏ
assert VERSION in ('T','B','L')
MODEL = ROOT/f'ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-{VERSION}.yaml'
assert MODEL.is_file(), MODEL
DATA_ROOT = Path('/kaggle/input/YOUR_DATASET')  # SỬA
TRAIN = 'images/train'  # SỬA
VAL = 'images/val'      # SỬA
NAMES = {0:'object'}    # SỬA tất cả tên lớp
assert (DATA_ROOT/TRAIN).is_dir(), DATA_ROOT/TRAIN
assert (DATA_ROOT/VAL).is_dir(), DATA_ROOT/VAL
assert sorted(NAMES)==list(range(len(NAMES))), 'Class ID phải từ 0 liên tiếp'
DATA = Path('/kaggle/working/mamba-data.yaml')
# JSON là YAML hợp lệ, không cần thư viện ngoài trong kernel notebook.
DATA.write_text(json.dumps(dict(path=str(DATA_ROOT),train=TRAIN,val=VAL,names=NAMES),ensure_ascii=False))
EPOCHS = 300  # default launcher của tác giả
BATCH = 4     # điều chỉnh Kaggle; launcher gốc mặc định 512
IMGSZ = 640
SEED = 0
PROJECT = '/kaggle/working/runs-mamba-original'


## 3. Kiểm tra mô hình gốc trước khi train
Không có CPU fallback trong source gốc. Cell này cần CUDA extension hoạt động, nếu lỗi phải dừng.

In [ ]:
probe = """
import sys,torch
from ultralytics import YOLO
model=YOLO(sys.argv[1]).model.cuda().eval()
assert any(type(m).__name__=='VSSBlock' for m in model.modules())
assert any(type(m).__name__=='XSSBlock' for m in model.modules())
with torch.no_grad():
    y=model(torch.rand(2,3,96,96,device='cuda'))
    pred=y[0] if isinstance(y,tuple) else y
    assert torch.isfinite(pred).all()
print('PASS original Mamba architecture:',sum(p.numel() for p in model.parameters()),'parameters')
"""
subprocess.run([PY,'-c',probe,str(MODEL)],check=True)


## 4. Train trực tiếp bằng API gốc
SGD, 300 epochs, lr0/lrf và augment lấy từ default.yaml của source. Chỉ thay batch/device/workers cho Kaggle và dataset của bạn. So sánh với YOLOv12s cần giữ cùng split, epochs, độ phân giải và cách đo; không suy ra model nào tốt hơn nếu chưa train.

In [ ]:
runner = """
import sys,json,subprocess
from pathlib import Path
from ultralytics import YOLO
model_path,data,epochs,batch,imgsz,seed,project,name=sys.argv[1:]
model=YOLO(model_path)
def metadata(trainer):
    import torch
    payload=dict(torch=torch.__version__,cuda=torch.version.cuda,
                 git_commit=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),
                 model=model_path,original_source=True)
    (trainer.save_dir/'notebook_environment.json').write_text(json.dumps(payload,indent=2))
model.add_callback('on_pretrain_routine_end',metadata)
model.train(data=data,epochs=int(epochs),batch=int(batch),imgsz=int(imgsz),seed=int(seed),
            optimizer='SGD',device='0',workers=2,amp=True,pretrained=False,
            project=project,name=name)
"""
subprocess.run([PY,'-c',runner,str(MODEL),str(DATA),str(EPOCHS),str(BATCH),str(IMGSZ),str(SEED),
                PROJECT,f'Mamba-YOLO-{VERSION}'],check=True)


## 5. Kết quả và tiếp tục run
Save Version kèm outputs. Run trùng tên có hậu tố. Resume dùng args đã lưu; khi đổi phiên Kaggle hãy tạo lại cùng đường dẫn /kaggle/working/mamba-data.yaml ở cell dataset. Chạy setup + dataset rồi resume, không chạy cell train mới.

In [ ]:
for p in sorted(Path(PROJECT).glob('*/weights/*.pt')):
    print(p)
# LAST='/kaggle/input/PREVIOUS_OUTPUT/last.pt'
# subprocess.run([PY,'-c',"import sys; from ultralytics import YOLO; YOLO(sys.argv[1]).train(resume=True)",LAST],check=True)
# BEST='/kaggle/working/runs-mamba-original/Mamba-YOLO-T/weights/best.pt'
# subprocess.run([PY,'-c',"import sys; from ultralytics import YOLO; YOLO(sys.argv[1]).val(data=sys.argv[2],device='0',batch=4)",BEST,str(DATA)],check=True)
